# Learning Vector Quantization (LVQ)

*Pure Python implementation from scratch — no external ML libraries.*

---

## What is LVQ?

**Learning Vector Quantization** is a prototype-based supervised classification algorithm invented by Teuvo Kohonen. It sits at the intersection of **nearest-neighbor methods** and **neural networks**:

- Like **KNN**, LVQ classifies a new point by finding the closest prototype (called a **codebook vector**) and assigning its label.
- Like **neural networks**, LVQ *learns* the prototypes through an iterative training process that adjusts their positions in feature space.

### Why use LVQ over KNN?

| | KNN | LVQ |
|---|---|---|
| Storage | Keeps *all* training data | Keeps only a small set of codebook vectors |
| Prediction speed | O(n × d) | O(m × d), where m ≪ n |
| Training | None | Iterative learning |
| Memory | High | Low |

### Key Concepts

- **Codebook vector**: A representative prototype in the same feature space as the training data, with the same number of attributes plus a class label.
- **BMU (Best Matching Unit)**: The codebook vector closest (by Euclidean distance) to a given input.
- **Competition mechanism**: If the BMU's class matches the input's class, the BMU moves *toward* the input. Otherwise, it moves *away*.
- **Learning rate decay**: The learning rate decreases over epochs to allow the model to converge.

### Algorithm Overview

```
1. Initialize codebook vectors (random samples from training data)
2. For each epoch:
   a. Decay the learning rate: rate = alpha × (1 - epoch / max_epochs)
   b. For each training instance:
      i.   Find the BMU among codebook vectors
      ii.  If BMU class == instance class:
               BMU = BMU + rate × (instance - BMU)     # move closer
      iii. Else:
               BMU = BMU - rate × (instance - BMU)     # move away
3. Return the learned codebook vectors
```

## Step 1: Euclidean Distance

We reuse the same distance function as in KNN. It computes the Euclidean distance between two vectors, excluding the last element (the class label):

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

In [ ]:
from math import sqrt

def calculate_euclidean_distance(row1, row2):
    distance = 0.0
    for i in range(len(row1) - 1):  # exclude last column (class label)
        distance += (row1[i] - row2[i]) ** 2
    return sqrt(distance)

# Toy dataset: [feature_1, feature_2, class_label]
dataset = [[1.80, 1.91, 0],
           [1.85, 2.11, 0],
           [2.31, 2.88, 0],
           [3.54, -3.21, 0],
           [3.66, 3.12, 0],
           [5.52, 2.13, 1],
           [6.32, 1.46, 1],
           [7.35, 2.34, 1],
           [7.78, 3.26, 1],
           [8.43, -0.34, 1]]

## Step 2: Find the Best Matching Unit (BMU)

Given a set of codebook vectors and a new input row, the **BMU** is the codebook vector with the smallest Euclidean distance to the input.

Steps:
1. Compute the distance between each codebook vector and the input.
2. Store results as `(codebook_vector, distance)` tuples.
3. Sort by distance and return the closest one.

In [ ]:
def calculate_BMU(codebooks, test_row):
    distances = list()
    for codebook in codebooks:
        dist = calculate_euclidean_distance(codebook, test_row)
        distances.append((codebook, dist))
    distances.sort(key=lambda every_tuple: every_tuple[1])
    return distances[0][0]

In [ ]:
test_row = dataset[0]
bmu = calculate_BMU(dataset, test_row)
print(bmu)

## Step 3: Initialize Random Codebook Vectors

Each codebook vector is created by randomly selecting feature values from existing training instances. This gives us a reasonable starting point in the feature space.

The codebook vector has the **same structure** as a training row: the same number of features plus a class label.

In [ ]:
from random import randrange

def make_random_codebook(train):
    n_index = len(train)
    n_features = len(train[0])
    # Pick a random value for each feature from a random training row
    codebook = [train[randrange(n_index)][i] for i in range(n_features)]
    return codebook

## Step 4: Train the Codebook Vectors

This is the core of LVQ. For each epoch:

1. **Decay the learning rate**: `rate = alpha × (1 - epoch / max_epochs)`.
   This gradually reduces the step size so the codebook vectors converge.

2. **For each training row**, find the BMU and update it:
   - If the BMU's class **matches** the training row's class → move the BMU **toward** the row:
     `bmu[i] += rate × (row[i] - bmu[i])`
   - If the BMU's class **differs** → move the BMU **away** from the row:
     `bmu[i] -= rate × (row[i] - bmu[i])`

3. Track the sum of squared errors to monitor convergence.

### Hyperparameters

| Parameter | Role | Typical values |
|---|---|---|
| `learning_rate` | Initial step size | 0.1 – 0.5 |
| `n_codebooks` | Number of prototype vectors | 1–5 per class |
| `epochs` | Number of full passes over the data | 50–500 |

In [ ]:
def train_codebooks(train, n_codebooks, learn_rate, epochs):
    codebooks = [make_random_codebook(train) for i in range(n_codebooks)]
    for epoch in range(epochs):
        rate = learn_rate * (1 - (epoch / float(epochs)))
        sum_error = 0.0
        for row in train:
            bmu = calculate_BMU(codebooks, row)
            for i in range(len(row) - 1):
                error = row[i] - bmu[i]
                sum_error += error ** 2
                if bmu[-1] == row[-1]:
                    bmu[i] += rate * error   # move toward the training instance
                else:
                    bmu[i] -= rate * error   # move away from the training instance
        print('Epoch [%d], Learning rate: [%.3f], Sum of squared errors: [%.3f]'
              % (epoch, rate, sum_error))
    return codebooks

### Run the training loop

We use 2 codebook vectors (one per class), a learning rate of 0.5, and 100 epochs.

In [ ]:
learning_rate = 0.5
n_epoch = 100
n_codebooks = 2
codebooks = train_codebooks(dataset, n_codebooks, learning_rate, n_epoch)
print('\nFinal codebook vectors: %s' % codebooks)

---

## Summary

| Aspect | Detail |
|---|---|
| **Type** | Prototype-based / competitive learning |
| **Related to** | KNN (nearest-neighbor), SOM (self-organizing maps) |
| **Training** | Iterative adjustment of codebook vectors |
| **Prediction** | Find BMU among codebook vectors, return its class |
| **Strengths** | Memory-efficient, fast prediction, interpretable prototypes |
| **Weaknesses** | Sensitive to initialization, requires learning rate tuning |

### Key Takeaways

1. LVQ compresses the training set into a small number of **codebook vectors** (prototypes).
2. The **competition mechanism** moves prototypes toward same-class instances and away from different-class instances.
3. **Learning rate decay** ensures convergence — large steps early for exploration, small steps later for fine-tuning.
4. At prediction time, classification is simply a **1-nearest-neighbor** lookup against the codebook vectors.

### Next Steps
- Apply LVQ to the Ionosphere dataset (see `case06_ionosphere_radar_LVQ.py`).
- Experiment with different numbers of codebook vectors per class.
- Compare LVQ accuracy with KNN on the same dataset.